# Random Forest 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.impute import SimpleImputer

In [2]:
# read the data

df_entrena = pd.read_csv('data/hoteles-entrena.csv')
df_prueba = pd.read_csv('data/hoteles-prueba.csv')

In [3]:
# Verificar valores faltantes y tipos de datos en df_entrena y df_prueba
print(df_entrena.info())
print(df_prueba.info())

# Eliminar la columna 'company' de df_entrena
df_entrena = df_entrena.drop(columns=['company'])
df_prueba = df_prueba.drop(columns=['company'])

# Eliminar la columna 'agent' de df_entrena y df_prueba
df_entrena.drop(columns=['agent'], inplace=True)
df_prueba.drop(columns=['agent'], inplace=True)

# Reemplazar valores 'none' por NaN (si aplica)
df_entrena.replace('none', pd.NA, inplace=True)
df_prueba.replace('none', pd.NA, inplace=True)

# Eliminar o imputar valores faltantes (en este ejemplo, eliminamos las filas con NaN)
df_entrena.dropna(inplace=True)
df_prueba.fillna(0, inplace=True)  # Rellenar con 0 en df_prueba si hay valores faltantes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52981 entries, 0 to 52980
Data columns (total 25 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           52981 non-null  object 
 1   lead_time                       52981 non-null  int64  
 2   stays_in_weekend_nights         52981 non-null  int64  
 3   stays_in_week_nights            52981 non-null  int64  
 4   adults                          52981 non-null  int64  
 5   children                        52981 non-null  object 
 6   meal                            52981 non-null  object 
 7   country                         52681 non-null  object 
 8   market_segment                  52981 non-null  object 
 9   distribution_channel            52981 non-null  object 
 10  is_repeated_guest               52981 non-null  int64  
 11  previous_cancellations          52981 non-null  int64  
 12  previous_bookings_not_canceled  

In [4]:
# Definir las columnas categóricas
categorical_columns = ['hotel', 'meal', 'country', 'market_segment', 'distribution_channel', 
                       'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']

# Convertir las columnas categóricas en variables dummy
df_entrena = pd.get_dummies(df_entrena, columns=categorical_columns, drop_first=True)
df_prueba = pd.get_dummies(df_prueba, columns=categorical_columns, drop_first=True)

# Asegurarse de que df_prueba tenga las mismas columnas que df_entrena (rellenar con 0 para las columnas que falten)
df_prueba = df_prueba.reindex(columns=df_entrena.columns, fill_value=0)



In [5]:
# Definir características (X) y la variable objetivo (y)
X = df_entrena.drop(columns=['children'])  # Ajusta la columna objetivo según tu dataset
y = df_entrena['children']  # Columna objetivo

# Dividir en conjunto de entrenamiento (80%) y validación (20%)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [7]:
# Verificar las columnas que contienen valores no numéricos
print(X_train.select_dtypes(include=['object']).head())



      required_car_parking_spaces arrival_date
10314                     parking   2016-07-03
17060                     parking   2017-04-13
11523                     parking   2016-08-22
18984                     parking   2017-07-02
13628                     parking   2016-11-18


In [8]:
# Reemplazar 'parking' con 1 (o 0 si es más adecuado)
df_entrena['required_car_parking_spaces'].replace('parking', 1, inplace=True)
df_prueba['required_car_parking_spaces'].replace('parking', 1, inplace=True)

# Convertir la columna a numérica, por si acaso contiene otros valores no numéricos
df_entrena['required_car_parking_spaces'] = pd.to_numeric(df_entrena['required_car_parking_spaces'], errors='coerce')
df_prueba['required_car_parking_spaces'] = pd.to_numeric(df_prueba['required_car_parking_spaces'], errors='coerce')


In [10]:
# Verificar los valores únicos en la columna 'required_car_parking_spaces'
print(df_entrena['required_car_parking_spaces'].unique())
print(df_prueba['required_car_parking_spaces'].unique())



[1]
[0 1]


In [11]:
# Verificar si aún hay columnas que no son numéricas en X_train
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        print(f"La columna {col} contiene valores no numéricos: {X_train[col].unique()}")


La columna required_car_parking_spaces contiene valores no numéricos: ['parking']
La columna arrival_date contiene valores no numéricos: ['2016-07-03' '2017-04-13' '2016-08-22' '2017-07-02' '2016-11-18'
 '2015-07-21' '2016-01-11' '2017-08-06' '2016-12-15' '2016-08-11'
 '2017-07-19' '2017-07-09' '2015-12-12' '2016-12-12' '2016-05-20'
 '2017-06-26' '2016-01-01' '2017-03-07' '2016-03-01' '2016-02-05'
 '2017-08-22' '2016-04-13' '2016-04-23' '2016-07-30' '2016-12-02'
 '2016-07-02' '2017-08-28' '2016-07-25' '2016-02-08' '2017-08-15'
 '2016-12-30' '2016-06-21' '2015-09-03' '2016-07-31' '2015-08-01'
 '2016-08-16' '2016-03-19' '2017-04-27' '2016-10-24' '2016-07-18'
 '2016-02-06' '2017-04-09' '2016-08-15' '2015-12-05' '2015-10-09'
 '2016-03-24' '2017-01-06' '2017-04-04' '2015-12-19' '2015-07-23'
 '2016-02-12' '2016-02-13' '2015-08-22' '2016-07-26' '2016-04-22'
 '2016-06-17' '2017-07-30' '2017-07-16' '2015-12-23' '2017-07-06'
 '2017-08-03' '2015-12-26' '2017-08-13' '2015-07-05' '2016-12-25'
 '201

In [12]:
# Reemplazar 'parking' con 1 en la columna required_car_parking_spaces
df_entrena['required_car_parking_spaces'] = df_entrena['required_car_parking_spaces'].replace('parking', 1)
df_prueba['required_car_parking_spaces'] = df_prueba['required_car_parking_spaces'].replace('parking', 1)

# Convertir la columna a tipo numérico (si aún no lo es)
df_entrena['required_car_parking_spaces'] = pd.to_numeric(df_entrena['required_car_parking_spaces'])
df_prueba['required_car_parking_spaces'] = pd.to_numeric(df_prueba['required_car_parking_spaces'])


In [13]:
# Convertir la columna arrival_date a datetime
df_entrena['arrival_date'] = pd.to_datetime(df_entrena['arrival_date'], errors='coerce')
df_prueba['arrival_date'] = pd.to_datetime(df_prueba['arrival_date'], errors='coerce')

# Extraer características adicionales si es necesario
# Por ejemplo, extraer el mes y el día de la semana
df_entrena['arrival_month'] = df_entrena['arrival_date'].dt.month
df_entrena['arrival_dayofweek'] = df_entrena['arrival_date'].dt.dayofweek

df_prueba['arrival_month'] = df_prueba['arrival_date'].dt.month
df_prueba['arrival_dayofweek'] = df_prueba['arrival_date'].dt.dayofweek

# Si no necesitas la columna original de arrival_date, puedes eliminarla
df_entrena = df_entrena.drop(columns=['arrival_date'])
df_prueba = df_prueba.drop(columns=['arrival_date'])


In [14]:
# Verificar si aún quedan columnas no numéricas
non_numeric_columns = df_entrena.select_dtypes(exclude=['float64', 'int64', 'uint8']).columns
print("Columnas no numéricas en df_entrena:", non_numeric_columns)

non_numeric_columns_prueba = df_prueba.select_dtypes(exclude=['float64', 'int64', 'uint8']).columns
print("Columnas no numéricas en df_prueba:", non_numeric_columns_prueba)


Columnas no numéricas en df_entrena: Index(['children'], dtype='object')
Columnas no numéricas en df_prueba: Index([], dtype='object')


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Separar características (X) y la variable objetivo (y) en df_entrena
X = df_entrena.drop(columns=['children'])
y = df_entrena['children']

# Dividir en entrenamiento y validación (80% entrenamiento, 20% validación)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Inicializar el escalador
scaler = StandardScaler()

# Escalar los datos de entrenamiento y validación
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Eliminar la columna 'children' de df_prueba antes de escalar
X_test = df_prueba.drop(columns=['children'], errors='ignore')

# Escalar el conjunto de prueba
X_test_scaled = scaler.transform(X_test)

